# Tau Intent Threshold Sweep

Notebook autocontido para calibracao de `tau_intent` usando apenas intent recognition.

Objetivos desta versao:
- usar apenas datasets publicos ja presentes no projeto
- amostrar `25` sentencas por `(dataset_name, semantic_intent)`
- rodar a parte pesada de inferencia uma unica vez
- salvar apenas `pred_intent` e `pred_intent_score` brutos
- fazer o sweep offline de `tau_intent` sobre esse cache local
- evitar qualquer alteracao em `src/`


In [ ]:
import sys
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

def find_repo_root(start: Path) -> Path:
    resolved = start.resolve()
    for candidate in [resolved, *resolved.parents]:
        if (candidate / "src").exists() and (candidate / "datasets").exists():
            return candidate
    raise RuntimeError(f"Could not find repository root starting from: {start}")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"REPO_ROOT = {REPO_ROOT}")

from src.core.pipeline import initialize_nlu
from src.data_tools.batch_eval_utils import (
    build_dataset_and_semantic_views,
    build_semantic_intent_candidates_by_dataset,
    build_semantic_intent_slot_dict,
    export_evaluation_csvs,
    load_semantic_alias_registry,
    merge_predictions_into_dataset_view,
    validate_registry_against_metadata,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


In [ ]:
RUN_ID = datetime.now().strftime("tau_intent_%Y%m%d_%H%M%S")
STUDY_TIMESTAMP = datetime.now().isoformat(timespec="seconds")

THRESHOLD_ROOT = REPO_ROOT / "threshold_evaluation"
RUN_DIR = THRESHOLD_ROOT / "runs" / RUN_ID
ARTIFACTS_DIR = RUN_DIR / "artifacts"
EVALUATION_DIR = RUN_DIR / "evaluation"

for path in (THRESHOLD_ROOT, RUN_DIR, ARTIFACTS_DIR, EVALUATION_DIR):
    path.mkdir(parents=True, exist_ok=True)

REGISTRY_PATH = REPO_ROOT / "datasets" / "metadata" / "semantic_alias_registry.json"
SAMPLING_SEED = 42
SAMPLES_PER_INTENT = 25
TAU_GRID = [round(step * 0.05, 2) for step in range(1, 20)]
UNKNOWN_INTENT_LABEL = "UNK_INTENT"
SLOT_DELIMITER = " || "
INTENT_BATCH_SIZE = 16
DEVICE = 0
REUSE_CACHED_TOP1_SCORES = True
FLOAT_FORMAT = "%.10f"

TOP1_CACHE_PATH = ARTIFACTS_DIR / "public_calibration_pool_top1_scores.csv"
SAMPLED_DATASET_PATH = ARTIFACTS_DIR / "sampled_dataset_view.csv"
SAMPLED_SEMANTIC_PATH = ARTIFACTS_DIR / "sampled_semantic_view.csv"
SAMPLE_COUNTS_PATH = ARTIFACTS_DIR / "sample_counts_by_dataset_intent.csv"
SWEEP_OVERALL_PATH = ARTIFACTS_DIR / "tau_intent_sweep_overall.csv"
SWEEP_BY_DATASET_PATH = ARTIFACTS_DIR / "tau_intent_sweep_by_dataset.csv"
SWEEP_BY_INTENT_PATH = ARTIFACTS_DIR / "tau_intent_sweep_by_intent.csv"
BEST_TAU_PATH = ARTIFACTS_DIR / "best_tau_selection.json"
STUDY_CONFIG_PATH = ARTIFACTS_DIR / "study_config.json"

config_preview = {
    "RUN_ID": RUN_ID,
    "RUN_DIR": str(RUN_DIR),
    "SAMPLING_SEED": SAMPLING_SEED,
    "SAMPLES_PER_INTENT": SAMPLES_PER_INTENT,
    "TAU_GRID": TAU_GRID,
    "INTENT_BATCH_SIZE": INTENT_BATCH_SIZE,
    "DEVICE": DEVICE,
    "TOP1_CACHE_PATH": str(TOP1_CACHE_PATH),
}
display(pd.DataFrame([config_preview]))


In [ ]:
PUBLIC_CALIBRATION_DATASET_PATHS = [
    REPO_ROOT / "datasets/snips/raw/train_AddToPlaylist_full.json",
    REPO_ROOT / "datasets/snips/raw/train_BookRestaurant_full.json",
    REPO_ROOT / "datasets/snips/raw/train_GetWeather_full.json",
    REPO_ROOT / "datasets/snips/processed/clean/train_PlayMusic_full_clean.json",
    REPO_ROOT / "datasets/snips/processed/clean/train_RateBook_full_clean.json",
    REPO_ROOT / "datasets/snips/processed/clean/train_SearchCreativeWork_full_clean.json",
    REPO_ROOT / "datasets/snips/processed/clean/train_SearchScreeningEvent_full_clean.json",
    REPO_ROOT / "datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json",
    REPO_ROOT / "datasets/multiwoz/processed/with_slots/slots_hospital__find_hospital.json",
    REPO_ROOT / "datasets/multiwoz/processed/with_slots/slots_hotel__find_hotel.json",
    REPO_ROOT / "datasets/multiwoz/processed/with_slots/slots_restaurant__find_restaurant.json",
    REPO_ROOT / "datasets/multiwoz/processed/with_slots/slots_taxi__find_taxi.json",
    REPO_ROOT / "datasets/multiwoz/processed/with_slots/slots_train__find_train.json",
    REPO_ROOT / "datasets/atis/processed/filtered/min50/abbreviation_min50.json",
    REPO_ROOT / "datasets/atis/processed/filtered/min50/aircraft_min50.json",
    REPO_ROOT / "datasets/atis/processed/filtered/min50/airfare_min50.json",
    REPO_ROOT / "datasets/atis/processed/filtered/min50/airline_min50.json",
    REPO_ROOT / "datasets/atis/processed/filtered/min50/flight_min50.json",
    REPO_ROOT / "datasets/atis/processed/filtered/min50/flight_time_min50.json",
    REPO_ROOT / "datasets/atis/processed/filtered/min50/ground_service_min50.json",
]

paths_df = pd.DataFrame({"dataset_path": [str(path) for path in PUBLIC_CALIBRATION_DATASET_PATHS]})
display(paths_df)


In [ ]:
registry = load_semantic_alias_registry(REGISTRY_PATH)
validate_registry_against_metadata(registry)

normalized_public_paths = [str(path).replace("\\", "/") for path in PUBLIC_CALIBRATION_DATASET_PATHS]
if any("/field_operations/" in path.lower() for path in normalized_public_paths):
    raise ValueError("field_operations nao deve entrar no pool publico de calibracao.")

dataset_view, semantic_view = build_dataset_and_semantic_views(
    PUBLIC_CALIBRATION_DATASET_PATHS,
    registry=registry,
    slot_delimiter=SLOT_DELIMITER,
)

dataset_names = sorted(dataset_view["dataset_name"].drop_duplicates().tolist())
semantic_intent_slot_dict = build_semantic_intent_slot_dict(dataset_names, registry=registry)
candidate_intents_by_dataset = build_semantic_intent_candidates_by_dataset(dataset_names, registry=registry)
candidate_intents = sorted(
    {
        candidate_intent
        for candidate_list in candidate_intents_by_dataset.values()
        for candidate_intent in candidate_list
    },
    key=lambda value: value.lower(),
)
candidate_summary = pd.DataFrame(
    [
        {
            "dataset_name": dataset_name,
            "candidate_count": len(candidate_list),
            "candidate_intents": " | ".join(candidate_list),
        }
        for dataset_name, candidate_list in candidate_intents_by_dataset.items()
    ]
).sort_values("dataset_name", kind="mergesort").reset_index(drop=True)
invalid_candidate_rows = semantic_view.loc[
    ~semantic_view.apply(
        lambda row: str(row["semantic_intent"]) in candidate_intents_by_dataset[str(row["dataset_name"])],
        axis=1,
    )
]
if any(not candidate_intents_by_dataset.get(dataset_name) for dataset_name in dataset_names):
    raise ValueError("Every dataset in the study must have a non-empty candidate intent list.")
if not invalid_candidate_rows.empty:
    raise ValueError("Found rows whose semantic_intent is not present in the candidate list of their dataset.")

print(f"dataset_view rows = {len(dataset_view)}")
print(f"semantic_view rows = {len(semantic_view)}")
print(f"dataset_names = {dataset_names}")
print(f"candidate_intents = {len(candidate_intents)}")
display(candidate_summary)

display(
    semantic_view[["dataset_name", "dataset_intent", "semantic_intent", "sentence"]]
    .head(10)
)


In [ ]:
MERGE_KEYS = ["example_id", "source_file", "dataset_name", "dataset_intent", "semantic_intent"]

def safe_div(numerator, denominator):
    return 0.0 if denominator == 0 else float(numerator) / float(denominator)

def to_serializable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {key: to_serializable(inner_value) for key, inner_value in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_serializable(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value

def normalize_classifier_outputs_local(outputs, expected_length):
    if isinstance(outputs, dict):
        outputs = [outputs]
    outputs = list(outputs or [])
    if len(outputs) != expected_length:
        raise ValueError(
            f"Expected {expected_length} classifier outputs, got {len(outputs)}."
        )
    return outputs

def batched(items, batch_size):
    if batch_size <= 0:
        raise ValueError("batch_size must be a positive integer.")
    for start in range(0, len(items), batch_size):
        yield start, items[start : start + batch_size]

def stratified_sample_semantic_view(semantic_view, sample_size=25, seed=42):
    sort_cols = ["dataset_name", "semantic_intent", "source_file", "example_id"]
    ordered = semantic_view.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)

    sampled_groups = []
    stats_rows = []
    group_cols = ["dataset_name", "semantic_intent"]

    for (dataset_name, semantic_intent), group_df in ordered.groupby(group_cols, sort=False, dropna=False):
        available_count = len(group_df)
        sampled_count = min(sample_size, available_count)
        shortfall_count = max(sample_size - available_count, 0)

        if sampled_count == available_count:
            sampled_group = group_df.copy()
        else:
            sampled_group = group_df.sample(n=sampled_count, random_state=seed)

        sampled_group = sampled_group.sort_values(sort_cols, kind="mergesort")
        sampled_groups.append(sampled_group)
        stats_rows.append(
            {
                "dataset_name": dataset_name,
                "semantic_intent": semantic_intent,
                "available_count": int(available_count),
                "sampled_count": int(sampled_count),
                "shortfall_count": int(shortfall_count),
            }
        )

    sampled_view = pd.concat(sampled_groups, ignore_index=True) if sampled_groups else ordered.iloc[0:0].copy()
    sampled_view = sampled_view.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)
    stats_view = pd.DataFrame(stats_rows).sort_values(group_cols, kind="mergesort").reset_index(drop=True)
    return sampled_view, stats_view

def run_batch_intent_top1_scores(
    nlu_processor,
    semantic_view,
    candidate_intents_by_dataset,
    batch_size=16,
    unknown_label="UNK_INTENT",
):
    if semantic_view.empty:
        empty = semantic_view.copy()
        empty["pred_intent"] = []
        empty["pred_intent_score"] = []
        empty["slot_eval_intent"] = []
        empty["pred_slots_semantic_json"] = []
        empty["pred_slots_score_json"] = []
        return empty

    working_view = semantic_view.copy()
    missing_datasets = sorted(
        {
            str(dataset_name)
            for dataset_name in working_view["dataset_name"].astype(str).unique().tolist()
            if str(dataset_name) not in candidate_intents_by_dataset
        }
    )
    if missing_datasets:
        raise KeyError(
            "Missing candidate intent lists for dataset(s): " + ", ".join(missing_datasets)
        )
    batch_frames = []

    for dataset_name, dataset_frame in working_view.groupby("dataset_name", sort=False):
        dataset_candidates = list(candidate_intents_by_dataset[str(dataset_name)])
        dataset_frame = dataset_frame.copy()
        sentences = dataset_frame["sentence"].astype(str).tolist()

        for start, batch_sentences in batched(sentences, batch_size):
            outputs = nlu_processor.models.classifier(
                batch_sentences,
                dataset_candidates,
                hypothesis_template="This command is intended to {}",
                multi_label=False,
                batch_size=batch_size,
            )
            outputs = normalize_classifier_outputs_local(outputs, len(batch_sentences))

            batch_view = dataset_frame.iloc[start : start + len(batch_sentences)].copy()
            pred_labels = []
            pred_scores = []

            for output in outputs:
                labels = output.get("labels", [])
                scores = output.get("scores", [])
                if not labels or not scores:
                    pred_labels.append(unknown_label)
                    pred_scores.append(0.0)
                    continue

                max_index = max(range(len(scores)), key=lambda idx: scores[idx])
                pred_labels.append(str(labels[max_index]))
                pred_scores.append(float(scores[max_index]))

            batch_view["pred_intent"] = pred_labels
            batch_view["pred_intent_score"] = pred_scores
            batch_view["slot_eval_intent"] = pred_labels
            batch_view["pred_slots_semantic_json"] = "{}"
            batch_view["pred_slots_score_json"] = "{}"
            batch_frames.append(batch_view)

    return pd.concat(batch_frames, ignore_index=False).sort_index(kind="mergesort").reset_index(drop=True)

def apply_tau_offline(
    df,
    tau_intent,
    pred_col="pred_intent",
    score_col="pred_intent_score",
    unknown_label="UNK_INTENT",
):
    output = df.copy()
    output["tau_intent"] = float(tau_intent)
    output["pred_intent_tau"] = np.where(
        output[score_col].astype(float) >= float(tau_intent),
        output[pred_col].astype(str),
        unknown_label,
    )
    return output

def compute_tau_metrics(
    df,
    gold_col="semantic_intent",
    pred_col="pred_intent_tau",
    unknown_label="UNK_INTENT",
):
    working = df.copy()
    gold_series = working[gold_col].astype(str)
    pred_series = working[pred_col].astype(str)

    total_count = int(len(working))
    covered_mask = pred_series != unknown_label
    covered_count = int(covered_mask.sum())
    abstained_count = int(total_count - covered_count)
    correct_covered = int(((gold_series == pred_series) & covered_mask).sum())

    labels = sorted(gold_series.dropna().unique().tolist(), key=lambda value: value.lower())
    per_label = {}
    macro_precision_values = []
    macro_recall_values = []
    macro_f1_values = []

    for label in labels:
        gold_is_label = gold_series == label
        pred_is_label = pred_series == label

        tp = int((gold_is_label & pred_is_label).sum())
        fp = int(((~gold_is_label) & pred_is_label).sum())
        fn = int((gold_is_label & (~pred_is_label)).sum())
        support = int(gold_is_label.sum())

        precision = safe_div(tp, tp + fp)
        recall = safe_div(tp, tp + fn)
        f1 = safe_div(2 * precision * recall, precision + recall) if (precision + recall) else 0.0

        per_label[label] = {
            "support": support,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
        macro_precision_values.append(precision)
        macro_recall_values.append(recall)
        macro_f1_values.append(f1)

    return {
        "total_count": total_count,
        "covered_count": covered_count,
        "abstained_count": abstained_count,
        "coverage": safe_div(covered_count, total_count),
        "abstention_rate": safe_div(abstained_count, total_count),
        "selective_accuracy": safe_div(correct_covered, covered_count),
        "macro_precision_global": float(np.mean(macro_precision_values)) if macro_precision_values else 0.0,
        "macro_recall_global": float(np.mean(macro_recall_values)) if macro_recall_values else 0.0,
        "macro_f1_global": float(np.mean(macro_f1_values)) if macro_f1_values else 0.0,
        "per_label": per_label,
    }

def compute_tau_metrics_by_group(
    df,
    tau_intent,
    group_cols,
    gold_col="semantic_intent",
    pred_col="pred_intent_tau",
    unknown_label="UNK_INTENT",
):
    rows = []
    for group_key, group_df in df.groupby(group_cols, sort=False, dropna=False):
        metrics = compute_tau_metrics(
            group_df,
            gold_col=gold_col,
            pred_col=pred_col,
            unknown_label=unknown_label,
        )
        row = {"tau_intent": float(tau_intent)}
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        row.update({column: value for column, value in zip(group_cols, group_key)})
        row.update({key: value for key, value in metrics.items() if key != "per_label"})
        rows.append(row)
    return pd.DataFrame(rows)

def compute_per_intent_metrics(
    df,
    tau_intent,
    group_cols=None,
    gold_col="semantic_intent",
    pred_col="pred_intent_tau",
    unknown_label="UNK_INTENT",
):
    group_cols = list(group_cols or [])
    rows = []

    if group_cols:
        grouped_iter = df.groupby(group_cols, sort=False, dropna=False)
    else:
        grouped_iter = [((), df)]

    for group_key, scope_df in grouped_iter:
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        base_payload = {"tau_intent": float(tau_intent)}
        base_payload.update({column: value for column, value in zip(group_cols, group_key)})

        gold_series = scope_df[gold_col].astype(str)
        pred_series = scope_df[pred_col].astype(str)
        labels = sorted(gold_series.dropna().unique().tolist(), key=lambda value: value.lower())

        for label in labels:
            gold_is_label = gold_series == label
            pred_is_label = pred_series == label

            tp = int((gold_is_label & pred_is_label).sum())
            fp = int(((~gold_is_label) & pred_is_label).sum())
            fn = int((gold_is_label & (~pred_is_label)).sum())
            support = int(gold_is_label.sum())
            covered_count = int((gold_is_label & (pred_series != unknown_label)).sum())
            abstained_count = int((gold_is_label & (pred_series == unknown_label)).sum())

            precision = safe_div(tp, tp + fp)
            recall = safe_div(tp, tp + fn)
            f1 = safe_div(2 * precision * recall, precision + recall) if (precision + recall) else 0.0

            row = dict(base_payload)
            row.update(
                {
                    "semantic_intent": label,
                    "support": support,
                    "covered_count": covered_count,
                    "abstained_count": abstained_count,
                    "precision": precision,
                    "recall": recall,
                    "f1": f1,
                    "tpr": recall,
                    "fdr": safe_div(fp, tp + fp),
                }
            )
            rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
sampled_semantic_view, sample_counts_by_dataset_intent = stratified_sample_semantic_view(
    semantic_view,
    sample_size=SAMPLES_PER_INTENT,
    seed=SAMPLING_SEED,
)

sample_keys = sampled_semantic_view[MERGE_KEYS].copy()
sample_keys["_sample_order"] = np.arange(len(sample_keys))

sampled_dataset_view = (
    dataset_view
    .merge(sample_keys, on=MERGE_KEYS, how="inner")
    .sort_values("_sample_order", kind="mergesort")
    .drop(columns=["_sample_order"])
    .reset_index(drop=True)
)
sampled_semantic_view = sampled_semantic_view.reset_index(drop=True)

if len(sampled_dataset_view) != len(sampled_semantic_view):
    raise ValueError(
        f"sampled_dataset_view ({len(sampled_dataset_view)}) and sampled_semantic_view ({len(sampled_semantic_view)}) must have the same number of rows."
    )

sampled_dataset_view.to_csv(SAMPLED_DATASET_PATH, index=False, encoding="utf-8-sig")
sampled_semantic_view.to_csv(SAMPLED_SEMANTIC_PATH, index=False, encoding="utf-8-sig")
sample_counts_by_dataset_intent.to_csv(SAMPLE_COUNTS_PATH, index=False, encoding="utf-8-sig")

written_gold_paths = export_evaluation_csvs(
    dataset_view=sampled_dataset_view,
    evaluation_root=EVALUATION_DIR,
)

print(f"sampled_dataset_view rows = {len(sampled_dataset_view)}")
print(f"sampled_semantic_view rows = {len(sampled_semantic_view)}")
display(sample_counts_by_dataset_intent)


In [ ]:
nlu_processor = initialize_nlu(device=DEVICE, tau_intent=0.2)

model_snapshot = {
    "nli_model_name": getattr(nlu_processor.models, "nli_model_name", None),
    "qa_model_name": getattr(nlu_processor.models, "qa_model_name", None),
    "spacy_model": getattr(nlu_processor.models, "spacy_model", None),
    "index_path": getattr(nlu_processor.models, "index_path", None),
    "device": getattr(nlu_processor.models, "device", None),
}
display(pd.DataFrame([model_snapshot]))


In [ ]:
if REUSE_CACHED_TOP1_SCORES and TOP1_CACHE_PATH.exists():
    sampled_results_view = pd.read_csv(
        TOP1_CACHE_PATH,
        encoding="utf-8-sig",
        keep_default_na=False,
        na_filter=False,
    )
    sampled_results_view["pred_intent_score"] = pd.to_numeric(
        sampled_results_view["pred_intent_score"],
        errors="coerce",
    ).fillna(0.0)
else:
    semantic_predictions = run_batch_intent_top1_scores(
        nlu_processor,
        sampled_semantic_view,
        candidate_intents_by_dataset,
        batch_size=INTENT_BATCH_SIZE,
        unknown_label=UNKNOWN_INTENT_LABEL,
    )

    sampled_results_view = merge_predictions_into_dataset_view(
        sampled_dataset_view,
        semantic_predictions,
    )
    sampled_results_view.to_csv(
        TOP1_CACHE_PATH,
        index=False,
        encoding="utf-8-sig",
        float_format=FLOAT_FORMAT,
    )

written_prediction_paths = export_evaluation_csvs(
    results_view=sampled_results_view,
    evaluation_root=EVALUATION_DIR,
)

print(f"Cached top-1 predictions at: {TOP1_CACHE_PATH}")
display(
    sampled_results_view[
        [
            "dataset_name",
            "semantic_intent",
            "sentence",
            "pred_intent",
            "pred_intent_score",
        ]
    ].head(10)
)


In [ ]:
overall_rows = []
by_dataset_frames = []
by_intent_frames = []

for tau_intent in TAU_GRID:
    tau_view = apply_tau_offline(
        sampled_results_view,
        tau_intent=tau_intent,
        pred_col="pred_intent",
        score_col="pred_intent_score",
        unknown_label=UNKNOWN_INTENT_LABEL,
    )

    overall_metrics = compute_tau_metrics(
        tau_view,
        gold_col="semantic_intent",
        pred_col="pred_intent_tau",
        unknown_label=UNKNOWN_INTENT_LABEL,
    )
    overall_rows.append(
        {
            "tau_intent": float(tau_intent),
            **{key: value for key, value in overall_metrics.items() if key != "per_label"},
        }
    )

    by_dataset_frames.append(
        compute_tau_metrics_by_group(
            tau_view,
            tau_intent=tau_intent,
            group_cols=["dataset_name"],
            gold_col="semantic_intent",
            pred_col="pred_intent_tau",
            unknown_label=UNKNOWN_INTENT_LABEL,
        )
    )
    by_intent_frames.append(
        compute_per_intent_metrics(
            tau_view,
            tau_intent=tau_intent,
            group_cols=["dataset_name"],
            gold_col="semantic_intent",
            pred_col="pred_intent_tau",
            unknown_label=UNKNOWN_INTENT_LABEL,
        )
    )

tau_intent_sweep_overall = pd.DataFrame(overall_rows).sort_values("tau_intent", kind="mergesort").reset_index(drop=True)
tau_intent_sweep_by_dataset = pd.concat(by_dataset_frames, ignore_index=True).sort_values(
    ["tau_intent", "dataset_name"],
    kind="mergesort",
).reset_index(drop=True)
tau_intent_sweep_by_intent = pd.concat(by_intent_frames, ignore_index=True).sort_values(
    ["tau_intent", "dataset_name", "semantic_intent"],
    kind="mergesort",
).reset_index(drop=True)

tau_intent_sweep_overall.to_csv(SWEEP_OVERALL_PATH, index=False, encoding="utf-8-sig", float_format=FLOAT_FORMAT)
tau_intent_sweep_by_dataset.to_csv(SWEEP_BY_DATASET_PATH, index=False, encoding="utf-8-sig", float_format=FLOAT_FORMAT)
tau_intent_sweep_by_intent.to_csv(SWEEP_BY_INTENT_PATH, index=False, encoding="utf-8-sig", float_format=FLOAT_FORMAT)

display(tau_intent_sweep_overall)


In [ ]:
tau_intent_sweep_overall = tau_intent_sweep_overall.copy()
tau_intent_sweep_overall["feasible"] = tau_intent_sweep_overall["coverage"] >= 0.98

best_feasible_df = tau_intent_sweep_overall.loc[tau_intent_sweep_overall["feasible"]].sort_values(
    ["macro_f1_global", "coverage", "selective_accuracy", "tau_intent"],
    ascending=[False, False, False, True],
    kind="mergesort",
).reset_index(drop=True)

if not best_feasible_df.empty:
    best_feasible_row = best_feasible_df.iloc[0].to_dict()
    best_infeasible_row = None
else:
    best_feasible_row = None
    best_infeasible_df = tau_intent_sweep_overall.sort_values(
        ["coverage", "macro_f1_global", "selective_accuracy", "tau_intent"],
        ascending=[False, False, False, True],
        kind="mergesort",
    ).reset_index(drop=True)
    best_infeasible_row = best_infeasible_df.iloc[0].to_dict() if not best_infeasible_df.empty else None

best_tau_selection = {
    "run_id": RUN_ID,
    "generated_at": STUDY_TIMESTAMP,
    "selection_rule": {
        "objective": "maximize_macro_f1_global",
        "coverage_constraint": 0.98,
        "feasible_sort_order": ["macro_f1_global desc", "coverage desc", "selective_accuracy desc", "tau_intent asc"],
        "infeasible_sort_order": ["coverage desc", "macro_f1_global desc", "selective_accuracy desc", "tau_intent asc"],
    },
    "best_feasible_tau": to_serializable(best_feasible_row),
    "best_infeasible_tau": to_serializable(best_infeasible_row),
}

study_config = {
    "run_id": RUN_ID,
    "generated_at": STUDY_TIMESTAMP,
    "repo_root": str(REPO_ROOT),
    "registry_path": str(REGISTRY_PATH),
    "sampling_seed": SAMPLING_SEED,
    "samples_per_intent": SAMPLES_PER_INTENT,
    "tau_grid": TAU_GRID,
    "unknown_intent_label": UNKNOWN_INTENT_LABEL,
    "slot_delimiter": SLOT_DELIMITER,
    "intent_batch_size": INTENT_BATCH_SIZE,
    "device": DEVICE,
    "reuse_cached_top1_scores": REUSE_CACHED_TOP1_SCORES,
    "dataset_paths": [str(path) for path in PUBLIC_CALIBRATION_DATASET_PATHS],
    "dataset_names": dataset_names,
    "candidate_intents": candidate_intents,
    "candidate_intents_by_dataset": candidate_intents_by_dataset,
    "sample_counts_by_dataset_intent": sample_counts_by_dataset_intent.to_dict(orient="records"),
    "artifacts": {
        "sampled_dataset_view": str(SAMPLED_DATASET_PATH),
        "sampled_semantic_view": str(SAMPLED_SEMANTIC_PATH),
        "sample_counts": str(SAMPLE_COUNTS_PATH),
        "top1_cache": str(TOP1_CACHE_PATH),
        "tau_sweep_overall": str(SWEEP_OVERALL_PATH),
        "tau_sweep_by_dataset": str(SWEEP_BY_DATASET_PATH),
        "tau_sweep_by_intent": str(SWEEP_BY_INTENT_PATH),
        "best_tau_selection": str(BEST_TAU_PATH),
    },
    "model_snapshot": to_serializable(model_snapshot),
    "written_gold_paths": to_serializable(written_gold_paths),
}

with BEST_TAU_PATH.open("w", encoding="utf-8") as fp:
    json.dump(best_tau_selection, fp, ensure_ascii=False, indent=2)

with STUDY_CONFIG_PATH.open("w", encoding="utf-8") as fp:
    json.dump(to_serializable(study_config), fp, ensure_ascii=False, indent=2)

summary_row = best_tau_selection["best_feasible_tau"] or best_tau_selection["best_infeasible_tau"]
display(pd.DataFrame([summary_row]) if summary_row else pd.DataFrame())
print(f"Saved best tau selection to: {BEST_TAU_PATH}")
print(f"Saved study config to: {STUDY_CONFIG_PATH}")


## Uso esperado

1. Rodar as celulas em ordem.
2. Conferir `sample_counts_by_dataset_intent.csv` para validar a amostragem.
3. Conferir `public_calibration_pool_top1_scores.csv` para o cache principal do sweep offline.
4. Usar `tau_intent_sweep_overall.csv` e `best_tau_selection.json` para escolher o threshold final.
